In [ ]:
# Potrzebny jest plik .env zawierający def. połączeń
from dotenv import load_dotenv

# Load connection variables into environment
load_dotenv()

In [ ]:
# JupySQL / SQLAlchemy connection
%reload_ext sql
%reload_ext mermaid_magic

# Read connection strings from environment variables
%sql $PG_CONN --alias pg
%sql $MSSQL_CONN --alias mssql
%sql $ORACLE_CONN --alias oracle

## PBD-2627 - Wprowadzenie: DDL i INSERT

**Zasady pracy**

- Komórka `%%sql pg` --> PostgreSQL, `%%sql mssql` --> SQL Server, `%%sql oracle` --> Oracle.
- Wszystkie nasze tabele mają prefiks `Lab` (`LabArtist`, `LabAlbum`…). **Nie dotykamy oryginalnych tabel Chinook.**
-  **Oczekiwany błąd** - komórka oznaczona tak *ma* zakończyć się błędem. W trybie „Run All” błąd zatrzyma wykonanie, więc uruchamiamy komórka po komórce.
- Powtarzasz notebook od początku? Najpierw uruchom sekcję **Sprzątanie** na końcu.
- Do przeczytania **Dodatek A** (karta dialektów).

In [ ]:
# Ograniczamy wyświetlanie wyników JupySQL (uruchom po komórkach z połączeniami)
%config SqlMagic.displaylimit = 10

## 1. Z kreatora Accessa do SQL

W Accessie każdy wybór w widoku projektu to w rzeczywistości **definicja struktury**. Dziś zapisujemy ją kodem.

| Access (widok projektu) | SQL |
|---|---|
| Autonumerowanie | `IDENTITY` (ANSI: `GENERATED … AS IDENTITY`) |
| Tekst krótki (255 zn.) | `VARCHAR(n)` - długość podajemy sami |
| Tekst długi | `TEXT` (PG) / `VARCHAR(MAX)` (MSSQL) / `CLOB` (Oracle, ANSI) |
| Liczba (liczba całkowita długa) | `INTEGER` |
| Waluta | `NUMERIC(p,s)` |
| Tak/Nie | `BOOLEAN` (PG) / `BIT` (MSSQL) / `BOOLEAN` od 23ai (Oracle) |
| Data/godzina | `TIMESTAMP` / `DATETIME2` (MSSQL) |
| Wymagane = Tak | `NOT NULL` |
| Reguła poprawności | `CHECK` |
| Wartość domyślna | `DEFAULT` |
| Indeksowane (bez duplikatów) | `UNIQUE` |
| Klucz podstawowy | `PRIMARY KEY` |
| Relacje + więzy integralności, kaskady | `FOREIGN KEY … ON DELETE …` |

**Pułapki dla osób z Accessa**

| Access | SQL (ANSI i nasze trzy systemy) |
|---|---|
| tekst w `"..."` | tekst w **`'...'`**; `"..."` to *identyfikator* |
| daty w `#2026-10-10#` | `DATE '2026-10-10'` (ANSI, PG, Oracle); T-SQL: `'2026-10-10'` |
| wieloznaczniki `*`, `?` | `%`, `_` (w `LIKE`) |
| `Prawda = -1` | `TRUE`/`FALSE` lub `1`/`0` zależnie od systemu |

```mermaid
flowchart LR
    C[CREATE TABLE] --> I[INSERT]
    I --> A[ALTER TABLE]
    A --> I
    A --> D[DROP TABLE]
```

Dzisiaj przejdziemy dokładnie tą ścieżką: **CREATE → INSERT → ALTER → DROP**.

## 2. Wprowadzenie

- Polecenie może zajmować wiele linii; kończymy je średnikiem.
- Wielkość liter w *słowach kluczowych* nie ma znaczenia; **w identyfikatorach - zależy od systemu** (tabela niżej).
- Tekst: `'apostrofy'`; apostrof w tekście podwajamy: `'Guns N'' Roses'`.
- Komentarze: `-- do końca linii` oraz `/* blokowy */`.
- Podwójny cudzysłów (ANSI) oznacza **identyfikator**, nie tekst. Nie używamy go w nazwach - unikamy kłopotów z wielkością liter.
- Nazywamy ograniczenia: `pk_…`, `fk_…`, `ck_…`, `uq_…`.

SQL jest ustandaryzowany (ANSI/ISO), ale konkretne systemy implementują standard (trochę po swojemu) oraz własne rozszerzenia.

```mermaid
flowchart TB
    S[SQL - standard] --> O[Oracle SQL]
    S --> M[Microsoft SQL Server / T-SQL]
    S --> P[PostgreSQL]
    O --> OP[PL/SQL]
    P --> PP[PL/pgSQL]
```

Dlatego w kursie będziemy rozróżniać:

- **SQL standard** czy **ANSI SQL** - składnia i semantyka wspólna dla wielu DBMS,
- **dialekt DBMS** - rozszerzenia i różnice implementacyjne,
- **język proceduralny** - np. PL/SQL, T-SQL, PL/pgSQL.

**Dialekty - tylko to, co ma znaczenie dla DDL**

| | PostgreSQL | SQL Server (T-SQL) | Oracle |
|---|---|---|---|
| Identyfikator bez cudzysłowu | → **małe** litery | zachowuje zapis; wielkość liter wg collation | → **WIELKIE** litery |
| Przestrzeń nazw | schemat (`public`) | schemat (`dbo`) | schemat = użytkownik |
| `''` (pusty tekst) | ≠ `NULL` | ≠ `NULL` | **= `NULL`** |
| DDL a transakcja | transakcyjne (`ROLLBACK` cofa) | transakcyjne | **niejawny `COMMIT`** |
| Autonumerowanie | `GENERATED … AS IDENTITY` | `IDENTITY(1,1)` | `GENERATED … AS IDENTITY` (12c+) |
| Tekst Unicode | `VARCHAR`/`TEXT` (UTF-8) | `NVARCHAR` + literał `N'…'` | `NVARCHAR2` lub `VARCHAR2` w AL32UTF8 |
| `TIMESTAMP` | data i czas |  **to nie data!** (rowversion) - używamy `DATETIME2` | data i czas |

## 3. Chinook „od środka” - co już jest zdefiniowane

Tabele, które zaraz sami odtworzymy (uproszczony fragment Chinook, z kolumnami i kluczami):

```mermaid
erDiagram
    ARTIST ||--o{ ALBUM : "ma"
    ALBUM  ||--o{ TRACK : "zawiera"
    GENRE  ||--o{ TRACK : "klasyfikuje"
    MEDIATYPE ||--o{ TRACK : "format"

    ARTIST {
        int ArtistId PK
        string Name
    }
    ALBUM {
        int AlbumId PK
        string Title
        int ArtistId FK
    }
    GENRE {
        int GenreId PK
        string Name
    }
    MEDIATYPE {
        int MediaTypeId PK
        string Name
    }
    TRACK {
        int TrackId PK
        string Name
        int AlbumId FK
        int MediaTypeId FK
        int GenreId FK
        string Composer
        int Milliseconds
        int Bytes
        decimal UnitPrice
    }
```

Zanim napiszemy własne `CREATE TABLE`, **odczytajmy cudzy**: jak DBMS opisuje kolumny i ograniczenia tabel `Track` i `Album`.
Każdy system robi to inaczej: PostgreSQL i SQL Server - `INFORMATION_SCHEMA` (standard), Oracle - własne widoki `USER_*`.

### 3.1 Kolumny tabeli `Track`

In [ ]:
%%sql pg
SELECT column_name, data_type, character_maximum_length AS len,
       numeric_precision AS prec, numeric_scale AS scale, is_nullable
FROM information_schema.columns
WHERE LOWER(table_name) = 'track'
ORDER BY ordinal_position;

In [ ]:
%%sql mssql
SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH AS len,
       NUMERIC_PRECISION AS prec, NUMERIC_SCALE AS scale, IS_NULLABLE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE LOWER(TABLE_NAME) = 'track'
ORDER BY ORDINAL_POSITION;

In [ ]:
%%sql oracle
SELECT column_name, data_type, data_length AS len,
       data_precision AS prec, data_scale AS scale, nullable
FROM user_tab_columns
WHERE UPPER(table_name) = 'TRACK'
ORDER BY column_id

Ten sam model, trzy różne zapisy typów (`NVARCHAR` / `VARCHAR2` / `character varying`, `numeric` / `NUMBER`). Zwróć uwagę na typ `UnitPrice` i `Milliseconds`.

### 3.2 Ograniczenia tabeli `Album`

In [ ]:
%%sql pg
SELECT constraint_name, constraint_type
FROM information_schema.table_constraints
WHERE LOWER(table_name) = 'album'
  AND constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY', 'UNIQUE');

In [ ]:
%%sql mssql
SELECT CONSTRAINT_NAME, CONSTRAINT_TYPE
FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS
WHERE LOWER(TABLE_NAME) = 'album'
  AND CONSTRAINT_TYPE IN ('PRIMARY KEY', 'FOREIGN KEY', 'UNIQUE');

In [ ]:
%%sql oracle
SELECT constraint_name, constraint_type
FROM user_constraints
WHERE UPPER(table_name) = 'ALBUM'
  AND constraint_type IN ('P', 'R', 'U')

W Oracle `P` = PRIMARY KEY, `R` = FOREIGN KEY (referential), `U` = UNIQUE. Dla kolumn ograniczeń: `USER_CONS_COLUMNS` (Oracle), `KEY_COLUMN_USAGE` (PG, MSSQL).

## 4. Typy danych - wybór wynika z semantyki

Najpierw pytamy: **co oznacza wartość?** - dopiero potem: **jak ją zapisać?**

| Chinook | Znaczenie | Typ | Dlaczego |
|---|---|---|---|
| `Track.Milliseconds` | czas trwania - liczba | `INTEGER` | liczymy, sumujemy |
| `Track.UnitPrice` | pieniądze | `NUMERIC(10,2)` | **nigdy** `FLOAT` - błędy zaokrągleń |
| `Customer.PostalCode`, `Phone` | identyfikator tekstowy | `VARCHAR(n)` | wiodące zera, `+48`, myślniki; nie liczymy |
| `Invoice.InvoiceDate` | moment w czasie | `TIMESTAMP` / `DATETIME2` | data **i** czas |
| `Track.Name` | tekst o ograniczonej długości | `VARCHAR(200)` | |

**Tabela porównawcza - ANSI jako punkt wyjścia** (wersje: PostgreSQL 18, SQL Server 2025, Oracle 23ai/26ai)

| Kategoria | ANSI SQL | PostgreSQL | SQL Server | Oracle |
|---|---|---|---|---|
| Liczba całkowita | `SMALLINT`, `INTEGER`, `BIGINT` | `SMALLINT`, `INTEGER`, `BIGINT` | `TINYINT`, `SMALLINT`, `INT`, `BIGINT` | `NUMBER(p,0)`; `INTEGER` = `NUMBER(38)` |
| Liczba **dokładna** | `NUMERIC(p,s)`, `DECIMAL(p,s)` | `NUMERIC(p,s)` | `NUMERIC(p,s)`, `DECIMAL(p,s)` | `NUMBER(p,s)` |
| Liczba **przybliżona** | `REAL`, `DOUBLE PRECISION`, `FLOAT` | `REAL`, `DOUBLE PRECISION` | `REAL`, `FLOAT` | `BINARY_FLOAT`, `BINARY_DOUBLE` |
| Tekst zmienny | `VARCHAR(n)` | `VARCHAR(n)`, `TEXT` | `VARCHAR(n)`, `NVARCHAR(n)` | `VARCHAR2(n)`, `NVARCHAR2(n)` |
| Tekst stały | `CHAR(n)` | `CHAR(n)` | `CHAR(n)` | `CHAR(n)` |
| Tekst duży | `CLOB` | `TEXT` | `VARCHAR(MAX)`, `NVARCHAR(MAX)` | `CLOB` |
| Data | `DATE` | `DATE` | `DATE` | `DATE` ( zawiera też czas) |
| Data i czas | `TIMESTAMP [WITH TIME ZONE]` | `TIMESTAMP`, `TIMESTAMPTZ` | `DATETIME2`, `DATETIMEOFFSET` | `TIMESTAMP [WITH TIME ZONE]` |
| Logiczny | `BOOLEAN` | `BOOLEAN` | `BIT` | `BOOLEAN` (od 23ai) |
| Binarny | `BLOB` | `BYTEA` | `VARBINARY(n/MAX)` | `BLOB`, `RAW(n)` |
| Autonumer | `GENERATED … AS IDENTITY` | `GENERATED … AS IDENTITY` (`SERIAL` - skrót) | `IDENTITY(1,1)` | `GENERATED … AS IDENTITY` (12c+) |

**Uwagi dialektowe**

- **SQL Server:** `VARCHAR` nie jest Unicode - dla polskich znaków użyj `NVARCHAR` i literału `N'…'`. Wersja Chinook dla MSSQL używa właśnie `NVARCHAR`.
- **Oracle:** `VARCHAR2(40)` to domyślnie 40 *bajtów* (nie znaków) - przy polskich znakach `VARCHAR2(40 CHAR)`. `INTEGER` jest aliasem `NUMBER(38)`. Zalecany jest `VARCHAR2`, nie `VARCHAR`.
- **PostgreSQL:** `TEXT` nie ma narzutu w porównaniu z `VARCHAR`; `SERIAL` to skrót, nie typ.

Tabela jest orientacyjna; pełne zestawienie - dokumentacja poszczególnych DBMS.

### Demo: dlaczego pieniądze nie w `FLOAT`

In [ ]:
%%sql pg
SELECT CASE WHEN CAST(0.1 AS DOUBLE PRECISION) + CAST(0.2 AS DOUBLE PRECISION) = CAST(0.3 AS DOUBLE PRECISION)
            THEN 'rowne' ELSE 'rozne' END AS przyblizone,
       CASE WHEN CAST(0.1 AS NUMERIC(10,2)) + CAST(0.2 AS NUMERIC(10,2)) = CAST(0.3 AS NUMERIC(10,2))
            THEN 'rowne' ELSE 'rozne' END AS dokladne;

In [ ]:
%%sql mssql
SELECT CASE WHEN CAST(0.1 AS DOUBLE PRECISION) + CAST(0.2 AS DOUBLE PRECISION) = CAST(0.3 AS DOUBLE PRECISION)
            THEN 'rowne' ELSE 'rozne' END AS przyblizone,
       CASE WHEN CAST(0.1 AS NUMERIC(10,2)) + CAST(0.2 AS NUMERIC(10,2)) = CAST(0.3 AS NUMERIC(10,2))
            THEN 'rowne' ELSE 'rozne' END AS dokladne;

In [ ]:
%%sql oracle
SELECT CASE WHEN CAST(0.1 AS BINARY_DOUBLE) + CAST(0.2 AS BINARY_DOUBLE) = CAST(0.3 AS BINARY_DOUBLE)
            THEN 'rowne' ELSE 'rozne' END AS przyblizone,
       CASE WHEN CAST(0.1 AS NUMBER(10,2)) + CAST(0.2 AS NUMBER(10,2)) = CAST(0.3 AS NUMBER(10,2))
            THEN 'rowne' ELSE 'rozne' END AS dokladne
FROM DUAL

Dwie różnice dialektowe „przy okazji”: Oracle (przed 23ai) wymaga `FROM DUAL`, a jego `FLOAT` jest zapisywany dziesiętnie - prawdziwą arytmetykę binarną daje `BINARY_DOUBLE`.

## 5. CREATE TABLE - od modelu do kodu

```sql
CREATE TABLE nazwa_tabeli (
    nazwa_kolumny  typ  [DEFAULT wartość]  [ograniczenie_kolumnowe ...],
    ...
    [CONSTRAINT nazwa] ograniczenie_tabelowe,
    ...
);
```

Kolejność w definicji kolumny: **typ → `DEFAULT` → ograniczenia** (Oracle wymaga, by `DEFAULT` poprzedzał ograniczenia).
Zapis `[…]` oznacza element opcjonalny.

### 5.1 Pierwsza tabela - ANSI, ten sam kod w trzech systemach

In [ ]:
%%sql pg
CREATE TABLE LabArtist (
    ArtistId INTEGER      NOT NULL,
    Name     VARCHAR(120) NOT NULL,
    CONSTRAINT pk_labartist PRIMARY KEY (ArtistId)
);

In [ ]:
%%sql mssql
CREATE TABLE LabArtist (
    ArtistId INTEGER      NOT NULL,
    Name     VARCHAR(120) NOT NULL,
    CONSTRAINT pk_labartist PRIMARY KEY (ArtistId)
);

In [ ]:
%%sql oracle
CREATE TABLE LabArtist (
    ArtistId INTEGER      NOT NULL,
    Name     VARCHAR(120) NOT NULL,
    CONSTRAINT pk_labartist PRIMARY KEY (ArtistId)
);

Ten sam kod zadziałał wszędzie - **ale co system z niego zrobił?** Sprawdźmy w Oracle:

In [ ]:
%%sql oracle
SELECT column_name, data_type, data_length, data_precision, nullable
FROM user_tab_columns
WHERE table_name = 'LABARTIST'
ORDER BY column_id

`INTEGER` → `NUMBER(38)`, `VARCHAR(120)` → `VARCHAR2(120)`, a nazwa tabeli → `LABARTIST`. To dobra ilustracja różnicy między **standardem a implementacją**.

### 5.2 Ograniczenia - przypisanie do rodzajów więzów

| Rodzaj więzów | Mechanizm | Uwagi dialektowe |
|---|---|---|
| obowiązkowość | `NOT NULL` | tylko na poziomie kolumny |
| unikalność | `UNIQUE` | SQL Server: **tylko jeden** `NULL` w kolumnie `UNIQUE`; PG i Oracle - wiele |
| dziedzinowe | typ, `CHECK`, `DEFAULT` | podzapytania w `CHECK` nie działają w żadnym z trzech systemów |
| integralność encji | `PRIMARY KEY` | |
| integralność referencyjna | `FOREIGN KEY … REFERENCES` | |
| propagacja | `ON DELETE {NO ACTION \| CASCADE \| SET NULL}` | `SET DEFAULT`, `RESTRICT` - nie wszędzie; `ON UPDATE` - nie w Oracle |

**Ograniczenie kolumnowe vs tabelowe.** Ograniczenie jednej kolumny można zapisać „przy kolumnie” albo osobno w tabeli:

```sql
-- kolumnowo
Name VARCHAR(120) NOT NULL UNIQUE
 -- tabelowo
 CONSTRAINT uq_name UNIQUE (Name)
 ```

Klucz **złożony** i ograniczenia obejmujące kilka kolumn - tylko tabelowo. `NOT NULL` - tylko kolumnowo.

Przykład `CREATE TABLE LabAlbum` zgodnie z diagramem ER: `AlbumId` (PK), `Title` (tekst, obowiązkowy, do 160 znaków), `ArtistId` (obowiązkowy, **klucz obcy** do `LabArtist`). Uruchom w jednym wybranym systemie:

In [ ]:
%%sql pg
CREATE TABLE LabAlbum (
    AlbumId  INTEGER      NOT NULL,
    Title    VARCHAR(160) NOT NULL,
    ArtistId INTEGER      NOT NULL,
    CONSTRAINT pk_labalbum PRIMARY KEY (AlbumId),
    CONSTRAINT fk_labalbum_artist FOREIGN KEY (ArtistId) REFERENCES LabArtist (ArtistId)
);

In [ ]:
%%sql mssql
CREATE TABLE LabAlbum (
    AlbumId  INTEGER      NOT NULL,
    Title    VARCHAR(160) NOT NULL,
    ArtistId INTEGER      NOT NULL,
    CONSTRAINT pk_labalbum PRIMARY KEY (AlbumId),
    CONSTRAINT fk_labalbum_artist FOREIGN KEY (ArtistId) REFERENCES LabArtist (ArtistId)
);

In [ ]:
%%sql oracle
CREATE TABLE LabAlbum (
    AlbumId  INTEGER      NOT NULL,
    Title    VARCHAR(160) NOT NULL,
    ArtistId INTEGER      NOT NULL,
    CONSTRAINT pk_labalbum PRIMARY KEY (AlbumId),
    CONSTRAINT fk_labalbum_artist FOREIGN KEY (ArtistId) REFERENCES LabArtist (ArtistId)
);

**Klucz obcy wskazuje od tabeli podrzędnej do nadrzędnej** - nie odwrotnie:

```mermaid
flowchart LR
    A["LabAlbum.ArtistId (FK)"] -->|REFERENCES| B["LabArtist.ArtistId (PK)"]
```

Dlatego tabelę nadrzędną tworzymy jako **pierwszą**.

### 5.3 `DEFAULT`, `CHECK` i klucz obcy z propagacją - `LabTrack`

In [ ]:
%%sql pg
CREATE TABLE LabTrack (
    TrackId      INTEGER       NOT NULL,
    Name         VARCHAR(200)  NOT NULL,
    AlbumId      INTEGER,
    Composer     VARCHAR(220),
    Milliseconds INTEGER       NOT NULL,
    UnitPrice    NUMERIC(10,2) DEFAULT 0.99 NOT NULL,
    CONSTRAINT pk_labtrack PRIMARY KEY (TrackId),
    CONSTRAINT fk_labtrack_album FOREIGN KEY (AlbumId) REFERENCES LabAlbum (AlbumId) ON DELETE SET NULL,
    CONSTRAINT ck_labtrack_ms    CHECK (Milliseconds > 0),
    CONSTRAINT ck_labtrack_price CHECK (UnitPrice >= 0)
);

In [ ]:
%%sql mssql
CREATE TABLE LabTrack (
    TrackId      INTEGER       NOT NULL,
    Name         VARCHAR(200)  NOT NULL,
    AlbumId      INTEGER,
    Composer     VARCHAR(220),
    Milliseconds INTEGER       NOT NULL,
    UnitPrice    NUMERIC(10,2) DEFAULT 0.99 NOT NULL,
    CONSTRAINT pk_labtrack PRIMARY KEY (TrackId),
    CONSTRAINT fk_labtrack_album FOREIGN KEY (AlbumId) REFERENCES LabAlbum (AlbumId) ON DELETE SET NULL,
    CONSTRAINT ck_labtrack_ms    CHECK (Milliseconds > 0),
    CONSTRAINT ck_labtrack_price CHECK (UnitPrice >= 0)
);

In [ ]:
%%sql oracle
CREATE TABLE LabTrack (
    TrackId      INTEGER       NOT NULL,
    Name         VARCHAR(200)  NOT NULL,
    AlbumId      INTEGER,
    Composer     VARCHAR(220),
    Milliseconds INTEGER       NOT NULL,
    UnitPrice    NUMERIC(10,2) DEFAULT 0.99 NOT NULL,
    CONSTRAINT pk_labtrack PRIMARY KEY (TrackId),
    CONSTRAINT fk_labtrack_album FOREIGN KEY (AlbumId) REFERENCES LabAlbum (AlbumId) ON DELETE SET NULL,
    CONSTRAINT ck_labtrack_ms    CHECK (Milliseconds > 0),
    CONSTRAINT ck_labtrack_price CHECK (UnitPrice >= 0)
);

To jest realizacja: **reguła biznesowa („cena nie może być ujemna”) żyje w DBMS i obowiązuje każdą aplikację**.

```mermaid
flowchart LR
    A1[Aplikacja A] --> DB[(DBMS)]
    A2[Aplikacja B] --> DB
    ETL[Proces ETL] --> DB
    DB --> C[Constraints]
    C --> OK[Integralne dane]
```

### 5.4 Autonumerowanie - składnia różni się (`LabGenre`)

ANSI: `GENERATED { ALWAYS | BY DEFAULT } AS IDENTITY` - PostgreSQL i Oracle. SQL Server: `IDENTITY(1,1)`.

In [ ]:
%%sql pg
CREATE TABLE LabGenre (
    GenreId INTEGER GENERATED BY DEFAULT AS IDENTITY NOT NULL,
    Name    VARCHAR(120) NOT NULL,
    CONSTRAINT pk_labgenre PRIMARY KEY (GenreId)
);

In [ ]:
%%sql mssql
CREATE TABLE LabGenre (
    GenreId INTEGER IDENTITY(1,1) NOT NULL,
    Name    VARCHAR(120) NOT NULL,
    CONSTRAINT pk_labgenre PRIMARY KEY (GenreId)
);

In [ ]:
%%sql oracle
CREATE TABLE LabGenre (
    GenreId INTEGER GENERATED BY DEFAULT AS IDENTITY NOT NULL,
    Name    VARCHAR2(120) NOT NULL,
    CONSTRAINT pk_labgenre PRIMARY KEY (GenreId)
)

`ALWAYS` odrzuca ręcznie podane wartości klucza; `BY DEFAULT` pozwala je podać (w SQL Server służy do tego `SET IDENTITY_INSERT`, w PostgreSQL przy `ALWAYS` - `OVERRIDING SYSTEM VALUE`).

### 5.5 Klucz złożony i `DEFAULT` na dacie - `LabTrackReview`

Typ „data i czas” i `DEFAULT CURRENT_TIMESTAMP` - tu **musimy** rozdzielić dialekty (T-SQL: `DATETIME2`, nie `TIMESTAMP`).

In [ ]:
%%sql pg
CREATE TABLE LabTrackReview (
    TrackId    INTEGER      NOT NULL,
    Reviewer   VARCHAR(60)  NOT NULL,
    Rating     INTEGER      NOT NULL,
    ReviewedAt TIMESTAMP    DEFAULT CURRENT_TIMESTAMP NOT NULL,
    CONSTRAINT pk_labtrackreview PRIMARY KEY (TrackId, Reviewer),
    CONSTRAINT fk_labtrackreview_track FOREIGN KEY (TrackId) REFERENCES LabTrack (TrackId) ON DELETE CASCADE,
    CONSTRAINT ck_labtrackreview_rating CHECK (Rating BETWEEN 1 AND 5)
);

In [ ]:
%%sql mssql
CREATE TABLE LabTrackReview (
    TrackId    INTEGER      NOT NULL,
    Reviewer   VARCHAR(60)  NOT NULL,
    Rating     INTEGER      NOT NULL,
    ReviewedAt DATETIME2    DEFAULT CURRENT_TIMESTAMP NOT NULL,
    CONSTRAINT pk_labtrackreview PRIMARY KEY (TrackId, Reviewer),
    CONSTRAINT fk_labtrackreview_track FOREIGN KEY (TrackId) REFERENCES LabTrack (TrackId) ON DELETE CASCADE,
    CONSTRAINT ck_labtrackreview_rating CHECK (Rating BETWEEN 1 AND 5)
);

In [ ]:
%%sql oracle
CREATE TABLE LabTrackReview (
    TrackId    INTEGER      NOT NULL,
    Reviewer   VARCHAR2(60) NOT NULL,
    Rating     INTEGER      NOT NULL,
    ReviewedAt TIMESTAMP    DEFAULT CURRENT_TIMESTAMP NOT NULL,
    CONSTRAINT pk_labtrackreview PRIMARY KEY (TrackId, Reviewer),
    CONSTRAINT fk_labtrackreview_track FOREIGN KEY (TrackId) REFERENCES LabTrack (TrackId) ON DELETE CASCADE,
    CONSTRAINT ck_labtrackreview_rating CHECK (Rating BETWEEN 1 AND 5)
)

Cały nowy model (tabele `Lab…`):

```mermaid
erDiagram
    LabArtist ||--o{ LabAlbum : "ma"
    LabAlbum  ||--o{ LabTrack : "zawiera"
    LabTrack  ||--o{ LabTrackReview : "ma recenzje"
    LabGenre
```

Dodatkowo:
- **`DEFERRABLE` / `INITIALLY DEFERRED`** - odroczone sprawdzanie ograniczeń (PostgreSQL, Oracle; SQL Server nie wspiera). 
    - W Oracle ograniczenie zadeklarowane jako `INITIALLY DEFERRED` jest odroczone bez `ALTER SESSION`; `SET CONSTRAINTS` / `ALTER SESSION SET CONSTRAINTS` zmienia tryb dla ograniczeń odraczalnych.
    - Domyślnie każde ograniczenie jest sprawdzane natychmiast, po każdym INSERT, UPDATE albo DELETE. Przy DEFERRABLE możesz to zmienić, a przy INITIALLY DEFERRED ograniczenie od razu startuje w trybie odroczonym. Takie ograniczenie można naruszyć w trakcie transakcji, byle było spełnione w chwili COMMIT.
    - Można zmienić tryb już w trakcie pracy, dla ograniczeń odraczalnych: 
        - SET CONSTRAINTS ALL DEFERRED (albo z nazwą ograniczenia) działa tylko w bieżącej transakcji.
        - ALTER SESSION SET CONSTRAINTS = DEFERRED | IMMEDIATE | DEFAULT działa w całej sesji. Ma to sens np. dla ograniczenia zadeklarowanego jako DEFERRABLE INITIALLY IMMEDIATE, które chcesz chwilowo odroczyć.
        - Ograniczenie zadeklarowane jako NOT DEFERRABLE (domyślne) nie da się odroczyć żadnym z tych poleceń.
    - Praktyczna różnica:
        - Przy DEFERRABLE INITIALLY IMMEDIATE ograniczenie zachowuje się domyślnie jak zwykłe. Odroczysz je dopiero świadomie, w konkretnej transakcji: SET CONSTRAINTS fk_labalbum_artist DEFERRED;.
        - Przy DEFERRABLE INITIALLY DEFERRED jest odwrotnie: ograniczenie jest odroczone od początku, a natychmiastowe sprawdzanie możesz włączyć poleceniem SET CONSTRAINTS … IMMEDIATE;. W obu wariantach COMMIT i tak wymusza sprawdzenie.
        - Pierwszy wariant jest bezpieczniejszy, bo błąd pojawia się natychmiast, w drugim zmienia się zachowanie dla wszystkich aplikacji zapisujących do tej tabeli, a błąd wychodzi dopiero przy COMMIT.
- **`CHECK` odwołujący się do innych kolumn** - zapisujemy jako ograniczenie *tabelowe* (ograniczenie kolumnowe odwołujące się do innej kolumny jest w niektórych systemach odrzucane).
- **Podzapytania w `CHECK`** - dozwolone przez standard, niezaimplementowane w PG, MSSQL ani Oracle; 
    - zamiast tego używa się wyzwalaczy.

## 6. INSERT - wypełniamy tabele

Dwie postaci polecenia:

```sql
INSERT INTO tabela (kolumna1, kolumna2, ...) VALUES (wartość1, wartość2, ...);     -- jawne wartości

INSERT INTO tabela (kolumna1, kolumna2, ...) SELECT kolumna1, kolumna2, ... FROM źródło;   -- z zapytania
```

**Zawsze podajemy listę kolumn** - kolejność kolumn nie ma znaczenia.

Możliwy jest zapis bez `INTO`: tylko T-SQL go dopuszcza; w ANSI, PostgreSQL i Oracle `INTO` jest obowiązkowe.

### 6.1 Jeden wiersz (ANSI - działa wszędzie)

In [ ]:
%%sql pg
INSERT INTO LabArtist (ArtistId, Name) VALUES (1001, 'Kult');

In [ ]:
%%sql mssql
INSERT INTO LabArtist (ArtistId, Name) VALUES (1001, 'Kult');

In [ ]:
%%sql oracle
INSERT INTO LabArtist (ArtistId, Name) VALUES (1001, 'Kult');

### 6.2 Wiele wierszy - tu dialekty się rozchodzą

ANSI (i PG, SQL Server 2008+, Oracle 23ai+): jedno `VALUES` z wieloma wierszami. Starsze Oracle: `INSERT ALL … SELECT … FROM DUAL`.

In [ ]:
%%sql pg
INSERT INTO LabArtist (ArtistId, Name)
VALUES (1002, 'Lady Pank'),
       (1003, 'Perfect');

In [ ]:
%%sql mssql
INSERT INTO LabArtist (ArtistId, Name)
VALUES (1002, 'Lady Pank'),
       (1003, 'Perfect');

In [ ]:
%%sql oracle
INSERT ALL
    INTO LabArtist (ArtistId, Name) VALUES (1002, 'Lady Pank')
    INTO LabArtist (ArtistId, Name) VALUES (1003, 'Perfect')
SELECT 1 FROM DUAL

### 6.3 Apostrof, Unicode i `NULL`

W Chinook jest wykonawca *Guns N' Roses* oraz *Björk*. Apostrof podwajamy; w SQL Server literał Unicode wymaga prefiksu `N`.

In [ ]:
%%sql pg
INSERT INTO LabArtist (ArtistId, Name)
VALUES (1004, 'Guns N'' Roses'),
       (1005, 'Björk');

In [ ]:
%%sql mssql
INSERT INTO LabArtist (ArtistId, Name)
VALUES (1004, N'Guns N'' Roses'),
       (1005, N'Björk'),
       (1006, N'łódź Kaliska');

In [ ]:
%%sql oracle
INSERT INTO LabArtist (ArtistId, Name)
VALUES (1002, 'Lady Pank'),
       (1003, 'Perfect');

W starszych wersjach Oracle (19c, 21c) takie VALUES kończy się błędem składni, więc tam INSERT ALL … SELECT 1 FROM DUAL jest jedynym sposobem na wstawienie kilku wierszy jednym poleceniem.

In [ ]:
%%sql oracle
INSERT ALL
    INTO LabArtist (ArtistId, Name) VALUES (1004, 'Guns N'' Roses')
    INTO LabArtist (ArtistId, Name) VALUES (1005, 'Björk')
SELECT 1 FROM DUAL

In [ ]:
%%sql mssql
-- VARCHAR w T-SQL nie jest Unicode: sprawdź, co stało się z polskimi znakami (wynik zależy od collation)
SELECT ArtistId, Name FROM LabArtist WHERE ArtistId >= 1004;

### 6.4 `INSERT … SELECT` - dane z oryginalnego Chinook

Kopiujemy fragment oryginalnych danych do naszych tabel; **kolejność wstawiania wynika z kluczy obcych** (rodzic przed dzieckiem).

In [ ]:
%%sql pg
INSERT INTO LabArtist (ArtistId, Name)
SELECT ArtistId, Name
FROM Artist
WHERE ArtistId <= 10;

In [ ]:
%%sql mssql
INSERT INTO LabArtist (ArtistId, Name)
SELECT ArtistId, Name
FROM Artist
WHERE ArtistId <= 10;

In [ ]:
%%sql oracle
INSERT INTO LabArtist (ArtistId, Name)
SELECT ArtistId, Name
FROM Artist
WHERE ArtistId <= 10;

In [ ]:
%%sql pg
INSERT INTO LabAlbum (AlbumId, Title, ArtistId)
SELECT AlbumId, Title, ArtistId
FROM Album
WHERE ArtistId <= 10;

In [ ]:
%%sql mssql
INSERT INTO LabAlbum (AlbumId, Title, ArtistId)
SELECT AlbumId, Title, ArtistId
FROM Album
WHERE ArtistId <= 10;

In [ ]:
%%sql oracle
INSERT INTO LabAlbum (AlbumId, Title, ArtistId)
SELECT AlbumId, Title, ArtistId
FROM Album
WHERE ArtistId <= 10;

In [ ]:
%%sql pg
INSERT INTO LabTrack (TrackId, Name, AlbumId, Composer, Milliseconds, UnitPrice)
SELECT TrackId, Name, AlbumId, Composer, Milliseconds, UnitPrice
FROM Track
WHERE AlbumId IN (SELECT AlbumId FROM LabAlbum);

In [ ]:
%%sql mssql
INSERT INTO LabTrack (TrackId, Name, AlbumId, Composer, Milliseconds, UnitPrice)
SELECT TrackId, Name, AlbumId, Composer, Milliseconds, UnitPrice
FROM Track
WHERE AlbumId IN (SELECT AlbumId FROM LabAlbum);

In [ ]:
%%sql oracle
INSERT INTO LabTrack (TrackId, Name, AlbumId, Composer, Milliseconds, UnitPrice)
SELECT TrackId, Name, AlbumId, Composer, Milliseconds, UnitPrice
FROM Track
WHERE AlbumId IN (SELECT AlbumId FROM LabAlbum);

### 6.5 Kopia tabeli: `CREATE TABLE … AS SELECT` i `SELECT … INTO`

Szybka kopia danych (bez ograniczeń!). PostgreSQL i Oracle: CTAS. SQL Server: `SELECT … INTO`.

In [ ]:
%%sql pg
CREATE TABLE LabGenreCopy AS SELECT * FROM Genre;

In [ ]:
%%sql mssql
SELECT * INTO LabGenreCopy FROM Genre;

In [ ]:
%%sql oracle
CREATE TABLE LabGenreCopy AS SELECT * FROM Genre

### 6.6 `DEFAULT` i autonumer - pomijamy kolumny

Nie podaliśmy `UnitPrice` - zadziała `DEFAULT 0.99`. Nie podajemy `GenreId` - numer nada system.

In [ ]:
%%sql pg
INSERT INTO LabTrack (TrackId, Name, Milliseconds) VALUES (9001, 'Utwór testowy', 180000);

In [ ]:
%%sql mssql
INSERT INTO LabTrack (TrackId, Name, Milliseconds) VALUES (9001, 'Utwór testowy', 180000);

In [ ]:
%%sql oracle
INSERT INTO LabTrack (TrackId, Name, Milliseconds) VALUES (9001, 'Utwór testowy', 180000);

In [ ]:
%%sql pg
INSERT INTO LabGenre (Name) VALUES ('Rock');

In [ ]:
%%sql mssql
INSERT INTO LabGenre (Name) VALUES ('Rock');

In [ ]:
%%sql oracle
INSERT INTO LabGenre (Name) VALUES ('Rock');

In [ ]:
%%sql pg
SELECT TrackId, Name, UnitPrice FROM LabTrack WHERE TrackId = 9001;

In [ ]:
%%sql pg
SELECT * FROM LabGenre;

### 6.7 Ograniczenia w akcji - cztery błędy (każdy w innym systemie)

 **Oczekiwane błędy.** Zwróć uwagę, jak różnie systemy je komunikują.

**Duplikat klucza głównego (PostgreSQL)**

In [ ]:
%%sql pg
INSERT INTO LabArtist (ArtistId, Name) VALUES (1, 'Duplikat');

**Naruszenie klucza obcego (SQL Server)**

In [ ]:
%%sql mssql
INSERT INTO LabAlbum (AlbumId, Title, ArtistId) VALUES (99999, 'Sierota', 99999);

**Naruszenie `CHECK` (Oracle)**

In [ ]:
%%sql oracle
INSERT INTO LabTrack (TrackId, Name, Milliseconds, UnitPrice) VALUES (9002, 'Ujemna cena', 1000, -1)

**Pusty tekst w kolumnie `NOT NULL` - ten sam kod, różne wyniki.** W PostgreSQL i SQL Server `''` to wartość (wstawienie się powiedzie); w Oracle `''` to `NULL` (błąd).

In [ ]:
%%sql pg
INSERT INTO LabArtist (ArtistId, Name) VALUES (2001, '');

In [ ]:
%%sql mssql
INSERT INTO LabArtist (ArtistId, Name) VALUES (2001, '');

In [ ]:
%%sql oracle
INSERT INTO LabArtist (ArtistId, Name) VALUES (2001, '')

Wiersz z pustą nazwą w PG i MSSQL zostaje w tabeli - `NOT NULL` **nie** chroni przed pustym tekstem. Do tego służy `CHECK (LENGTH(Name) > 0)`.

## 7. ALTER TABLE - struktura też się zmienia

Mamy już **dane**, więc zmiany muszą być zgodne z istniejącymi wierszami.

| Operacja | ANSI / PostgreSQL | SQL Server | Oracle |
|---|---|---|---|
| dodaj kolumnę | `ADD COLUMN c typ` | `ADD c typ` | `ADD (c typ)` |
| zmień typ / długość | `ALTER COLUMN c TYPE typ` | `ALTER COLUMN c typ [NOT] NULL` | `MODIFY (c typ)` |
| usuń kolumnę | `DROP COLUMN c` | `DROP COLUMN c` | `DROP COLUMN c` |
| zmień nazwę kolumny | `RENAME COLUMN a TO b` | `sp_rename 'T.a', 'b', 'COLUMN'` | `RENAME COLUMN a TO b` |
| dodaj ograniczenie | `ADD CONSTRAINT …` | `ADD CONSTRAINT …` | `ADD CONSTRAINT …` |
| usuń ograniczenie | `DROP CONSTRAINT nazwa` | `DROP CONSTRAINT nazwa` | `DROP CONSTRAINT nazwa` |

### 7.1 Dodanie kolumny (nullowalnej)

In [ ]:
%%sql pg
ALTER TABLE LabArtist ADD COLUMN Country CHAR(2);

In [ ]:
%%sql mssql
ALTER TABLE LabArtist ADD Country CHAR(2);

In [ ]:
%%sql oracle
ALTER TABLE LabArtist ADD (Country CHAR(2))

### 7.2 Kolumna obowiązkowa do tabeli z danymi

 **Oczekiwany błąd:** istniejące wiersze nie mają jak spełnić `NOT NULL`.

In [ ]:
%%sql pg
ALTER TABLE LabArtist ADD COLUMN IsActive CHAR(1) NOT NULL;

Rozwiązanie: `DEFAULT` wypełni istniejące wiersze.

In [ ]:
%%sql pg
ALTER TABLE LabArtist ADD COLUMN IsActive CHAR(1) DEFAULT 'Y' NOT NULL;

In [ ]:
%%sql mssql
ALTER TABLE LabArtist ADD IsActive CHAR(1) DEFAULT 'Y' NOT NULL;

In [ ]:
%%sql oracle
ALTER TABLE LabArtist ADD (IsActive CHAR(1) DEFAULT 'Y' NOT NULL)

### 7.3 Dodanie ograniczenia (ANSI - wszędzie tak samo)

Przy istniejących danych system **najpierw sprawdzi wszystkie wiersze**.

In [ ]:
%%sql pg
ALTER TABLE LabArtist ADD CONSTRAINT ck_labartist_active CHECK (IsActive IN ('Y', 'N'));

In [ ]:
%%sql mssql
ALTER TABLE LabArtist ADD CONSTRAINT ck_labartist_active CHECK (IsActive IN ('Y', 'N'));

In [ ]:
%%sql oracle
ALTER TABLE LabArtist ADD CONSTRAINT ck_labartist_active CHECK (IsActive IN ('Y', 'N'));

### 7.4 Zmiana typu i długości - poszerzenie i zawężenie

T-SQL: **powtarzamy `NOT NULL`** - inaczej kolumna stanie się nullowalna!

In [ ]:
%%sql pg
ALTER TABLE LabArtist ALTER COLUMN Name TYPE VARCHAR(200);

In [ ]:
%%sql mssql
ALTER TABLE LabArtist ALTER COLUMN Name VARCHAR(200) NOT NULL;

In [ ]:
%%sql oracle
ALTER TABLE LabArtist MODIFY (Name VARCHAR2(200))

 **Oczekiwany błąd:** zawężenie poniżej długości istniejących danych.

In [ ]:
%%sql pg
ALTER TABLE LabArtist ALTER COLUMN Name TYPE VARCHAR(5);

### 7.5 Zmiana nazwy i usunięcie kolumny

In [ ]:
%%sql pg
ALTER TABLE LabArtist RENAME COLUMN Country TO CountryCode;

In [ ]:
%%sql mssql
EXEC sp_rename 'LabArtist.Country', 'CountryCode', 'COLUMN';

In [ ]:
%%sql oracle
ALTER TABLE LabArtist RENAME COLUMN Country TO CountryCode

In [ ]:
%%sql pg
ALTER TABLE LabArtist DROP COLUMN CountryCode;

In [ ]:
%%sql mssql
ALTER TABLE LabArtist DROP COLUMN CountryCode;

In [ ]:
%%sql oracle
ALTER TABLE LabArtist DROP COLUMN CountryCode;

Zmiana nazwy to w T-SQL nie `ALTER`, ale procedura składowana `sp_rename`. Usunięcie kolumny jest **nieodwracalne** i niszczy jej dane.

## 8. DROP TABLE - operacja destrukcyjna

`DROP TABLE` usuwa **obiekt schematu razem z danymi** - to nie „czyszczenie tabeli”.

```sql
DROP TABLE nazwa_tabeli [CASCADE | RESTRICT];   -- ANSI
```

| | PostgreSQL | SQL Server | Oracle |
|---|---|---|---|
| tabela, do której odwołują się FK | błąd; `CASCADE` usuwa **ograniczenia FK** innych tabel | błąd; **brak** `CASCADE` - najpierw usuń FK / tabele podrzędne | błąd (ORA-02449); `CASCADE CONSTRAINTS` |
| `IF EXISTS` | tak | tak (2016+) | od 23ai |
| odzyskanie | `ROLLBACK` (DDL transakcyjne) | `ROLLBACK` (DDL transakcyjne) | kosz: `FLASHBACK TABLE … TO BEFORE DROP`; `PURGE` usuwa trwale |

### 8.1 Rodzic przed dzieckiem? Nie - dziecko przed rodzicem

 **Oczekiwany błąd** - `LabArtist` jest wskazywany przez `LabAlbum`:

In [ ]:
%%sql pg
DROP TABLE LabArtist;

```mermaid
flowchart TB
    X[DROP TABLE] --> Q{Czy inne tabele mają FK do tej tabeli?}
    Q -->|tak| R[Usuń najpierw tabele podrzędne lub ich FK]
    Q -->|nie| D[Usuń tabelę]
    R --> D
```

Kolejność usuwania jest **odwrotna** do kolejności tworzenia: `LabTrackReview → LabTrack → LabAlbum → LabArtist`.

### 8.2 DDL a transakcja

PostgreSQL i SQL Server: `DROP TABLE` można wycofać. Oracle: DDL kończy się niejawnym `COMMIT`.

> Jeśli JupySQL w Twojej konfiguracji nie utrzyma transakcji między poleceniami, pokaż ten fragment w `psql` / SSMS.

In [ ]:
%%sql pg
BEGIN;
DROP TABLE LabGenreCopy;
ROLLBACK;

In [ ]:
%%sql pg
SELECT COUNT(*) AS liczba_wierszy FROM LabGenreCopy;

In [ ]:
%%sql mssql
BEGIN TRAN;
DROP TABLE LabGenreCopy;
ROLLBACK;

In [ ]:
%%sql mssql
SELECT COUNT(*) AS liczba_wierszy FROM LabGenreCopy;

### 8.3 Oracle - kosz zamiast `ROLLBACK`

In [ ]:
%%sql oracle
DROP TABLE LabGenreCopy

In [ ]:
%%sql oracle
FLASHBACK TABLE LabGenreCopy TO BEFORE DROP

In [ ]:
%%sql oracle
SELECT COUNT(*) AS liczba_wierszy FROM LabGenreCopy

Aby **nie** zostawiać tabeli w koszu: `DROP TABLE … PURGE`.

### 8.4 Propagacja przy usuwaniu wiersza (zapowiedź DML)

`ON DELETE SET NULL` w `LabTrack` i `ON DELETE CASCADE` w `LabTrackReview` - zobaczmy to w praktyce (`DELETE` poznamy na kolejnych zajęciach):

In [ ]:
%%sql pg
INSERT INTO LabTrackReview (TrackId, Reviewer, Rating) VALUES (9001, 'Anna', 5);

In [ ]:
%%sql mssql
INSERT INTO LabTrackReview (TrackId, Reviewer, Rating) VALUES (9001, 'Anna', 5);

In [ ]:
%%sql oracle
INSERT INTO LabTrackReview (TrackId, Reviewer, Rating) VALUES (9001, 'Anna', 5);

In [ ]:
%%sql pg
DELETE FROM LabTrack WHERE TrackId = 9001;

In [ ]:
%%sql mssql
DELETE FROM LabTrack WHERE TrackId = 9001;

In [ ]:
%%sql oracle
DELETE FROM LabTrack WHERE TrackId = 9001;

In [ ]:
%%sql pg
SELECT COUNT(*) AS recenzje_po_usunieciu_utworu FROM LabTrackReview WHERE TrackId = 9001;

## 9. Podsumowanie

| Temat | ANSI (baza) | Co się zmienia w dialektach |
|---|---|---|
| `CREATE TABLE` | kolumny, typy, `CONSTRAINT` | nazwy typów, `IDENTITY`, `DATETIME2`, `DEFAULT` przed ograniczeniami |
| ograniczenia | `PRIMARY KEY`, `FOREIGN KEY`, `UNIQUE`, `CHECK`, `NOT NULL`, `DEFAULT` | `UNIQUE`+`NULL` w T-SQL; `''` = `NULL` w Oracle; `ON UPDATE` |
| `INSERT` | `INTO (kolumny) VALUES / SELECT` | wiele wierszy (`INSERT ALL`), literały (`N'…'`, `DATE '…'`) |
| `ALTER TABLE` | `ADD / DROP CONSTRAINT`, `DROP COLUMN` | `ADD`, `MODIFY`, `ALTER COLUMN`, zmiana nazwy |
| `DROP TABLE` | `DROP TABLE [CASCADE]` | `CASCADE CONSTRAINTS`, `PURGE`, brak `CASCADE` w T-SQL, `IF EXISTS` |

### Znajdź błąd (każdy kod pochodzi z innego dialektu)

```sql
-- a) uruchomiono w PostgreSQL
ALTER TABLE LabArtist ADD (Country CHAR(2));

-- b) uruchomiono w SQL Server
CREATE TABLE LabGenre2 (GenreId INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY);

-- c) uruchomiono w Oracle (starsza wersja)
INSERT INTO LabArtist (ArtistId, Name) VALUES (3001, 'A'), (3002, 'B');
```

## Sprzątanie

Usuwamy tabele w kolejności **od podrzędnych do nadrzędnych**.
Uruchom przed ponownym przejściem przez notebook (błędy „tabela nie istnieje” w tej sekcji są nieszkodliwe).

In [ ]:
%%sql pg
DROP TABLE IF EXISTS LabTrackReview;
DROP TABLE IF EXISTS LabTrack;
DROP TABLE IF EXISTS LabAlbum;
DROP TABLE IF EXISTS LabArtist;
DROP TABLE IF EXISTS LabGenre;
DROP TABLE IF EXISTS LabGenreCopy;

In [ ]:
%%sql mssql
DROP TABLE IF EXISTS LabTrackReview;
DROP TABLE IF EXISTS LabTrack;
DROP TABLE IF EXISTS LabAlbum;
DROP TABLE IF EXISTS LabArtist;
DROP TABLE IF EXISTS LabGenre;
DROP TABLE IF EXISTS LabGenreCopy;

W 23ai+ działa analogiczne: **DROP TABLE IF EXISTS**, poniżej jest klasyczna wersja: **DROP TABLE … PURGE** bez **IF EXISTS**.

In [ ]:
%%sql oracle
DROP TABLE LabTrackReview PURGE;
DROP TABLE LabTrack PURGE;
DROP TABLE LabAlbum PURGE;
DROP TABLE LabArtist PURGE;
DROP TABLE LabGenre PURGE;
DROP TABLE LabGenreCopy PURGE

## Dodatek A. Karta dialektów (pre-read, tylko to, co dotyczy DDL i INSERT)

**Oracle (Oracle SQL / PL/SQL)**
- Schemat = użytkownik (`CREATE USER`); odwołanie `schemat.obiekt`.
- Identyfikatory bez cudzysłowu → wielkie litery; `"..."` rozróżnia wielkość liter.
- `''` = `NULL`.
- Typy: `VARCHAR2`, `NUMBER`, `DATE` (z czasem); `BOOLEAN` od 23ai; `VARCHAR2(n)` to domyślnie bajty.
- Klucze: `GENERATED … AS IDENTITY` (12c+) lub `SEQUENCE`.
- Każde DDL kończy transakcję (niejawny `COMMIT`); transakcja startuje niejawnie.
- Wiele wierszy w `INSERT`: `INSERT ALL`; od 23ai także wielowierszowe `VALUES`.
- `DROP TABLE … CASCADE CONSTRAINTS [PURGE]`; kosz (`FLASHBACK TABLE`).

**SQL Server (T-SQL)**
- Instancja → bazy → schematy (domyślnie `dbo`) → obiekty.
- Ograniczniki `[nazwa]` (lub `"nazwa"` przy `QUOTED_IDENTIFIER ON`); wielkość liter wg collation.
- `''` ≠ `NULL`; w `UNIQUE` tylko jeden `NULL`.
- Typy: `NVARCHAR` + `N'…'`, `BIT`, `DATETIME2`, `UNIQUEIDENTIFIER`; `TIMESTAMP` = rowversion.
- Klucze: `IDENTITY(1,1)`; ręczne wartości: `SET IDENTITY_INSERT`.
- Domyślnie autocommit; jawne transakcje `BEGIN TRAN`; DDL jest transakcyjne.
- Brak `DROP TABLE … CASCADE`; zmiana nazw przez `sp_rename`; `SELECT … INTO` jako CTAS.

**PostgreSQL (SQL / PL/pgSQL)**
- Baza → schematy (domyślnie `public`; od wersji 15 zwykły użytkownik nie tworzy tam tabel bez nadanych uprawnień).
- Identyfikatory bez cudzysłowu → małe litery.
- `''` ≠ `NULL`.
- Typy: `TEXT`, `BOOLEAN`, `TIMESTAMPTZ`, `JSONB`, tablice; `SERIAL` lub `GENERATED … AS IDENTITY`.
- DDL jest transakcyjne.
- `DROP TABLE … CASCADE`; `RETURNING`, `INSERT … ON CONFLICT` (poza zakresem dzisiejszych zajęć).